# Homework 2 — Convolutional Neural Networks

This notebook mirrors the four sections of the PDF:

1. **Convolution Fundamentals** — parameter counting, manual 2D convolution, output-size formula, edge detectors, padding, receptive field.
2. **Pooling** — max vs average pooling, effect on receptive field.
3. **Architecture Analysis** — LeNet-5 parameter accounting, why 3×3 filters dominate.
4. **Backpropagation Through Conv Layers** — gradients with respect to kernel and input, max- and avg-pool gradients (sparse vs dense).

A final **Bonus** section trains a small CNN on MNIST end-to-end.

All numerical answers should match the PDF exactly. Run the notebook top-to-bottom; outputs are reproducible thanks to fixed random seeds.

In [ ]:
# ============================================================================
# Imports and reproducibility
# ============================================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Fixed seeds so all outputs (random pixels, weight init, etc.) are reproducible.
np.random.seed(42)
torch.manual_seed(42)

# Slightly nicer numpy / torch printing.
np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3)

print(f"PyTorch: {torch.__version__}, NumPy: {np.__version__}")


---
## Section 1 — Convolution Fundamentals


### Problem 1 — FC vs Conv parameter counting [5 pts]

Compare a fully-connected layer mapping a flattened $32 \times 32 \times 3$ image to 128 hidden units versus a Conv2d layer with 16 filters of size $5 \times 5$.

**Expected answers** (from the PDF):
- FC: $128 \cdot 3072 + 128 = 393{,}344$
- Conv: $16 \cdot (5 \cdot 5 \cdot 3 + 1) = 1{,}216$
- Ratio: $\approx 323\times$


In [ ]:
def count_params(module: nn.Module) -> int:
    # Total number of learnable parameters in a module.
    return sum(p.numel() for p in module.parameters())

# (a) Fully-connected: flatten 32x32x3 -> 128 hidden units
fc_layer = nn.Linear(32 * 32 * 3, 128)
fc_params = count_params(fc_layer)
print(f"Fully-connected layer:        {fc_params:>10,d} parameters")

# (b) Conv2d: 16 filters of size 5x5x3 (3 input channels, 16 output channels)
conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5)
conv_params = count_params(conv_layer)
print(f"Conv2d (16 filters of 5x5):  {conv_params:>10,d} parameters")

# (c) Ratio
print(f"\nRatio: {fc_params / conv_params:.1f}x more parameters in FC")


**Why this works:** Locality + weight sharing. The FC layer connects every output to every input pixel; the Conv layer reuses the same $5 \times 5$ filter at every spatial position. Parameter count for Conv depends only on filter size and channel counts — *not* on the image's spatial dimensions.

### Problem 2 — 2D convolution by hand [8 pts]

Implement 2D cross-correlation from scratch with nested loops, then verify against `torch.nn.functional.conv2d`.

**Setup** (matches the PDF's example):
$$
\mathbf{X} = \begin{pmatrix} 1 & 2 & 3 & 0 & 1 \\ 0 & 1 & 2 & 3 & 1 \\ 2 & 1 & 0 & 1 & 2 \\ 1 & 0 & 2 & 3 & 0 \\ 0 & 1 & 1 & 2 & 1 \end{pmatrix}, \quad
\mathbf{K} = \begin{pmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{pmatrix}
$$


In [ ]:
def conv2d_naive(X: np.ndarray, K: np.ndarray, stride: int = 1, padding: int = 0) -> np.ndarray:
    # 2D cross-correlation (the deep-learning convention -- no kernel flipping).
    # Returns Y of shape ((H + 2P - F)/S + 1, (W + 2P - F)/S + 1).
    if padding > 0:
        X = np.pad(X, padding, mode="constant", constant_values=0)
    H, W = X.shape
    F = K.shape[0]
    H_out = (H - F) // stride + 1
    W_out = (W - F) // stride + 1
    Y = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            patch = X[i * stride : i * stride + F, j * stride : j * stride + F]
            Y[i, j] = (patch * K).sum()
    return Y


X = np.array([
    [1, 2, 3, 0, 1],
    [0, 1, 2, 3, 1],
    [2, 1, 0, 1, 2],
    [1, 0, 2, 3, 0],
    [0, 1, 1, 2, 1],
], dtype=float)

K = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
], dtype=float)

# (a) Stride 1, no padding
Y = conv2d_naive(X, K, stride=1, padding=0)
print("Output (stride=1, padding=0):")
print(Y)
print(f"Shape: {Y.shape}")


In [ ]:
# (b) Verify against PyTorch's conv2d (this should pass once you fill in conv2d_naive)
X_t = torch.from_numpy(X).float().reshape(1, 1, 5, 5)
K_t = torch.from_numpy(K).float().reshape(1, 1, 3, 3)
Y_torch = F.conv2d(X_t, K_t, stride=1, padding=0).squeeze().numpy()

print("PyTorch output:")
print(Y_torch)
try:
    print(f"\nManual matches PyTorch: {np.allclose(Y, Y_torch)}")
except NameError:
    print("\nDefine Y in the previous cell first.")


In [ ]:
# (c) Stride 2 with padding 1
Y2 = conv2d_naive(X, K, stride=2, padding=1)
print("Output (stride=2, padding=1):")
print(Y2)
print(f"Shape: {Y2.shape} -- formula: floor((5 + 2 - 3)/2) + 1 = 3")


### Problem 3 — Output size formula [4 pts]

Verify the formula
$$ W_{\text{out}} = \left\lfloor \frac{W_{\text{in}} + 2P - F}{S} \right\rfloor + 1 $$
on several test cases, then compute the parameter count for a multi-channel layer.


In [ ]:
def output_size(W: int, F: int, P: int, S: int) -> int:
    return (W + 2 * P - F) // S + 1

# Test cases from the PDF
test_cases = [
    # (W,   F, P, S, expected)
    (28,   3, 0, 1, 26),
    (28,   3, 1, 1, 28),
    (28,   5, 2, 1, 28),
    (224,  3, 1, 2, 112),
    (224,  7, 3, 2, 112),
]
print(f"{'W':>4} {'F':>3} {'P':>3} {'S':>3}   {'computed':>9} {'expected':>9}  match")
print("-" * 50)
for W, F_, P, S, expected in test_cases:
    out = output_size(W, F_, P, S)
    print(f"{W:>4} {F_:>3} {P:>3} {S:>3}   {out:>9} {expected:>9}  {out == expected}")


In [ ]:
def conv_params(F: int, C_in: int, C_out: int) -> int:
    # Parameters in a Conv2d layer with FxF kernel, including biases.
    return F * F * C_in * C_out + C_out

# ResNet-style first layer: 7x7 kernel, 3 -> 64
n = conv_params(F=7, C_in=3, C_out=64)
print(f"ResNet first layer (7x7, 3 -> 64):  {n:,} parameters")
# Verify against PyTorch
layer = nn.Conv2d(3, 64, kernel_size=7)
print(f"PyTorch Conv2d:                     {count_params(layer):,} parameters")


### Problem 4 — Edge detector demo [4 pts]

Hand-design vertical and horizontal edge detector kernels (Sobel-style) and apply them to a synthetic image with both kinds of edges.


In [ ]:
# Synthetic image: bright on the right, slightly brighter on the bottom
img = np.zeros((20, 20))
img[:, 10:] = 1.0          # vertical edge at column 10
img[10:, :] = img[10:, :] + 0.5   # horizontal edge at row 10

# Sobel-style edge detectors
K_v = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=float)            # responds to vertical edges (intensity changes left-to-right)

K_h = K_v.T                 # responds to horizontal edges

out_v = conv2d_naive(img, K_v)
out_h = conv2d_naive(img, K_h)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1.5)
axes[0].set_title("Input image")
axes[1].imshow(out_v, cmap="RdBu_r")
axes[1].set_title("Vertical edge detector")
axes[2].imshow(out_h, cmap="RdBu_r")
axes[2].set_title("Horizontal edge detector")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()
print("\nNotice: the vertical-edge detector lights up at column ~10,")
print("the horizontal-edge detector lights up at row ~10. Each filter")
print("only responds to its own kind of intensity change.")


### Problem 5 — Padding modes [4 pts]

Compare *valid*, *same*, and *full* convolutions on the same input. For stride 1 and odd kernel size $F$, *same* padding requires $P = (F-1)/2$.


In [ ]:
# Same input, three different padding choices
np.random.seed(0)
X = np.random.rand(8, 8)
F_size = 3
K_uniform = np.ones((F_size, F_size)) / F_size**2  # box blur

Y_valid = conv2d_naive(X, K_uniform, padding=0)             # P = 0
Y_same  = conv2d_naive(X, K_uniform, padding=(F_size - 1) // 2)  # P = 1
Y_full  = conv2d_naive(X, K_uniform, padding=F_size - 1)    # P = 2

print(f"Input shape:  {X.shape}")
print(f"Valid (P=0):  output {Y_valid.shape}  (W - F + 1 = 6)")
print(f"Same  (P=1):  output {Y_same.shape}   (preserves input size!)")
print(f"Full  (P=2):  output {Y_full.shape}  (W + F - 1 = 10)")


### Problem 6 — Receptive field walkthrough [4 pts]

The receptive field of a neuron at layer $\ell$ is governed by the recurrence:
$$
j_\ell = j_{\ell-1} \cdot S_\ell, \qquad
\text{RF}_\ell = \text{RF}_{\ell-1} + (F_\ell - 1) \cdot j_{\ell-1}
$$
with $j_0 = 1$ and $\text{RF}_0 = 1$. Pooling layers double $j$, which then *multiplies* the receptive field gain of all subsequent conv layers.


In [ ]:
def compute_rf(layers):
    # layers: list of (F, S) tuples.
    # Returns: a list of dicts logging j and RF at every layer.
    j, rf = 1, 1
    history = [{"layer": 0, "op": "input",       "j": j, "RF": rf}]
    for idx, (F_l, S_l) in enumerate(layers, start=1):
        rf = rf + (F_l - 1) * j
        j  = j * S_l
        history.append({"layer": idx, "op": f"F={F_l}, S={S_l}", "j": j, "RF": rf})
    return history

# 5-layer network from the PDF:
#  Conv 5x5  -> Pool 2x2 s=2 -> Conv 5x5 -> Pool 2x2 s=2 -> Conv 3x3
network = [(5, 1), (2, 2), (5, 1), (2, 2), (3, 1)]
history = compute_rf(network)

print(f"{'Layer':>5}  {'Operation':<14}  {'j':>3}  {'RF':>3}")
print("-" * 35)
for h in history:
    print(f"{h['layer']:>5}  {h['op']:<14}  {h['j']:>3}  {h['RF']:>3}")
print(f"\nFinal receptive field: {history[-1]['RF']} x {history[-1]['RF']} pixels")


---
## Section 2 — Pooling


### Problem 7 — Max vs average pooling [4 pts]

Apply $2 \times 2$ max and average pooling (stride 2) to a $4 \times 4$ matrix, then verify with PyTorch.


In [ ]:
X = np.array([
    [2, 5, 1, 3],
    [4, 1, 8, 0],
    [7, 2, 1, 4],
    [3, 0, 6, 9],
], dtype=float)

def maxpool2x2(X):
    H, W = X.shape
    out = np.zeros((H // 2, W // 2))
    for i in range(H // 2):
        for j in range(W // 2):
            out[i, j] = X[i*2 : i*2+2, j*2 : j*2+2].max()
    return out

def avgpool2x2(X):
    H, W = X.shape
    out = np.zeros((H // 2, W // 2))
    for i in range(H // 2):
        for j in range(W // 2):
            out[i, j] = X[i*2 : i*2+2, j*2 : j*2+2].mean()
    return out

print("Max pool (manual):")
print(maxpool2x2(X))
print("\nAvg pool (manual):")
print(avgpool2x2(X))

# Verify with PyTorch
X_t = torch.from_numpy(X).float().reshape(1, 1, 4, 4)
print("\nMax pool (PyTorch):")
print(F.max_pool2d(X_t, 2).squeeze().numpy())
print("\nAvg pool (PyTorch):")
print(F.avg_pool2d(X_t, 2).squeeze().numpy())


**When to use which.** Max pool is the standard choice for detection-like tasks (it preserves the strongest local activation). Average pool is preferred for the *final* aggregation in modern architectures (e.g., global average pooling in ResNet) because it retains more information.

### Problem 8 — Pooling effect on receptive field [4 pts]

A network of two conv layers, then a max-pool, then two more conv layers. Reuse the recurrence from Problem 6.


In [ ]:
# Conv 3x3 -> Conv 3x3 -> MaxPool 2x2 (s=2) -> Conv 3x3 -> Conv 3x3
network = [(3, 1), (3, 1), (2, 2), (3, 1), (3, 1)]
history = compute_rf(network)

print(f"{'Layer':>5}  {'Operation':<14}  {'j':>3}  {'RF':>3}")
print("-" * 35)
for h in history:
    print(f"{h['layer']:>5}  {h['op']:<14}  {h['j']:>3}  {h['RF']:>3}")
print(f"\nFinal receptive field: {history[-1]['RF']} x {history[-1]['RF']}")
print("\nObservation: the maxpool only added 1 to RF directly, but DOUBLED")
print("the jump factor j. Subsequent conv layers now contribute 4 to RF")
print("each (not 2), because each kernel step covers 2 input pixels.")


---
## Section 3 — Architecture Analysis


### Problem 9 — LeNet-5 parameter count [7 pts]

Build the original LeNet-5 in PyTorch and verify the **61,706** parameter total. Expected breakdown:
- Conv $5\times5$, 1 → 6:  $5\cdot 5\cdot 1\cdot 6 + 6 = 156$
- Conv $5\times5$, 6 → 16: $5\cdot 5\cdot 6\cdot 16 + 16 = 2{,}416$
- FC $400 \to 120$: $48{,}120$
- FC $120 \to 84$:  $10{,}164$
- FC $84 \to 10$:    $850$

Total: 61,706.


In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.AvgPool2d(2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.AvgPool2d(2, stride=2)
        self.fc1   = nn.Linear(16 * 5 * 5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        # Input expected: (B, 1, 32, 32)
        x = self.pool1(torch.tanh(self.conv1(x)))   # -> (B, 6, 14, 14)
        x = self.pool2(torch.tanh(self.conv2(x)))   # -> (B, 16, 5, 5)
        x = x.flatten(1)                             # -> (B, 400)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.fc3(x)

lenet = LeNet5()

print("LeNet-5 parameter breakdown:")
print("-" * 50)
total = 0
for name, layer in lenet.named_children():
    n = count_params(layer)
    print(f"  {name:>6}: {n:>10,} parameters")
    total += n
print("-" * 50)
print(f"  {'TOTAL':>6}: {total:>10,} parameters")
print(f"\nMatches PDF expected value (61,706): {total == 61706}")


In [ ]:
# Compare with an MLP that achieves similar MNIST accuracy
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(32 * 32, 300), nn.Tanh(),
    nn.Linear(300, 300),     nn.Tanh(),
    nn.Linear(300, 10),
)
mlp_params = count_params(mlp)
print(f"Equivalent MLP (1024 -> 300 -> 300 -> 10): {mlp_params:,} parameters")
print(f"MLP / LeNet-5 ratio: {mlp_params / total:.2f}x more parameters in the MLP")


### Problem 10 — Why $3 \times 3$ filters dominate [7 pts]

Compare two architectures with the **same** receptive field of $5 \times 5$:
- A: one $5 \times 5$ conv, $C \to C$
- B: two stacked $3 \times 3$ convs, each $C \to C$

B has fewer parameters (about 72% of A) and twice as many nonlinearities (two ReLUs instead of one), for the same RF.


In [ ]:
C = 64  # channels in and out

# Architecture A: a single 5x5 convolution
arch_a = nn.Conv2d(C, C, kernel_size=5, padding=2)
params_a = count_params(arch_a)

# Architecture B: two stacked 3x3 convolutions
arch_b = nn.Sequential(
    nn.Conv2d(C, C, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(C, C, kernel_size=3, padding=1),
)
params_b = count_params(arch_b)

print(f"Architecture A (1x 5x5 conv):  {params_a:>8,} parameters, RF = 5")
print(f"Architecture B (2x 3x3 convs): {params_b:>8,} parameters, RF = 5")
print(f"\nB uses {100 * params_b / params_a:.1f}% of A's parameters,")
print(f"yet has 2 ReLU nonlinearities (vs 1) for the same effective RF.")
print(f"Saving: {100 * (1 - params_b / params_a):.0f}% fewer parameters.")


In [ ]:
# Generalize: stack n 3x3 convs has RF = 2n + 1
# Compare with one (2n+1) x (2n+1) conv at the same RF.
print(f"{'n':>2}  {'RF':>3}  {'stacked 3x3 params':>20}  {'single conv params':>20}  {'savings':>10}")
print("-" * 65)
for n in [1, 2, 3, 4, 5]:
    rf = 2 * n + 1
    p_stacked = n * (3 * 3 * C * C)         # bias ignored for simplicity
    p_single  = rf * rf * C * C
    savings = 100 * (1 - p_stacked / p_single)
    print(f"{n:>2}  {rf:>3}  {p_stacked:>20,}  {p_single:>20,}  {savings:>9.0f}%")


---
## Section 4 — Backpropagation Through Conv Layers


### Problem 11 — Gradient with respect to the kernel [10 pts]

For a 1D convolution $y_i = \sum_{u} x_{i+u} K_u$, the gradient with respect to the kernel is itself a convolution:
$$
\frac{\partial \mathcal{L}}{\partial K_u} = \sum_i \delta_i \, x_{i+u}
\qquad \text{where} \qquad \delta_i = \frac{\partial \mathcal{L}}{\partial y_i}
$$

Compute this manually, then verify against PyTorch's autograd.


In [ ]:
# Tiny example: x of length 5, kernel of length 3
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
K = np.array([0.5, -0.5, 1.0])
n, k = len(x), len(K)

# Forward: y has length n - k + 1 = 3
y = np.zeros(n - k + 1)
for i in range(n - k + 1):
    y[i] = sum(x[i + u] * K[u] for u in range(k))
print(f"Forward output y = {y}")

# Suppose the upstream gradient (from a downstream loss) is delta = [1, 1, 1].
# This is what you get if the loss is loss = sum(y).
delta = np.ones(n - k + 1)

# Manual gradient w.r.t. kernel: dL/dK_u = sum_i delta_i * x_{i+u}
dK_manual = np.zeros(k)
for u in range(k):
    dK_manual[u] = sum(delta[i] * x[i + u] for i in range(n - k + 1))
print(f"\nManual  dL/dK = {dK_manual}")


In [ ]:
# Verify against PyTorch autograd (this should pass once your manual code is correct)
x_t = torch.tensor([[[1.0, 2.0, 3.0, 4.0, 5.0]]])               # (B=1, C_in=1, L=5)
K_t = torch.tensor([[[0.5, -0.5, 1.0]]], requires_grad=True)    # (C_out=1, C_in=1, L=3)

y_t = F.conv1d(x_t, K_t)
loss = y_t.sum()       # this gives upstream gradient = ones at every position
loss.backward()

dK_torch = K_t.grad.squeeze().numpy()
print(f"PyTorch dL/dK = {dK_torch}")
try:
    print(f"\nManual matches PyTorch: {np.allclose(dK_manual, dK_torch)}")
except NameError:
    print("\nDefine dK_manual in the previous cell first.")


**Why this matters:** The fact that the kernel gradient is itself a convolution is exactly why PyTorch's `Conv2d.backward()` is implemented as another convolution call. The forward and backward operations have the same structure — only the operands change.

### Problem 12 — Max-pool gradient (sparse) [5 pts]

The max-pool gradient flows **only through the argmax position** of each window. Every other position in a window receives zero gradient. This is in contrast to conv layers, where the gradient is dense.


In [ ]:
X = torch.tensor([[[
    [1.0, 3.0, 2.0, 0.0],
    [5.0, 4.0, 1.0, 6.0],
    [0.0, 2.0, 7.0, 3.0],
    [1.0, 1.0, 5.0, 8.0],
]]], requires_grad=True)  # shape (1, 1, 4, 4) -- leaf tensor so X.grad works

y = F.max_pool2d(X, kernel_size=2, stride=2)
print("Max pool output:")
print(y.squeeze().detach().numpy())

# Upstream gradient (from the PDF): [[1, 2], [3, 4]]
delta = torch.tensor([[[[1.0, 2.0],
                        [3.0, 4.0]]]])
y.backward(gradient=delta)

print("\nGradient w.r.t. input:")
print(X.grad.squeeze().numpy())
print("\nObserve the SPARSITY: each 2x2 window has exactly ONE nonzero")
print("position --- the argmax. The other three receive zero gradient.")


### Problem 13 — Avg-pool gradient (dense) [5 pts]

For $k \times k$ average pooling with stride $k$, every input position in a window gets $1/k^2$ of that window's upstream gradient. The gradient is dense.


In [ ]:
# Same input, but use average pool this time
X = torch.tensor([[[
    [1.0, 3.0, 2.0, 0.0],
    [5.0, 4.0, 1.0, 6.0],
    [0.0, 2.0, 7.0, 3.0],
    [1.0, 1.0, 5.0, 8.0],
]]], requires_grad=True)  # shape (1, 1, 4, 4) -- leaf tensor so X.grad works

y = F.avg_pool2d(X, kernel_size=2, stride=2)
print("Avg pool output:")
print(y.squeeze().detach().numpy())

delta = torch.tensor([[[[1.0, 2.0],
                        [3.0, 4.0]]]])
y.backward(gradient=delta)

print("\nGradient w.r.t. input:")
print(X.grad.squeeze().numpy())
print("\nEvery position in each 2x2 window gets 1/4 of its window's upstream value.")
print("Compare: top-left window receives 1/4 = 0.25 at every position;")
print("         bottom-right window receives 4/4 = 1.0 at every position.")


**Practical consequence.** Max pool can leave "dead" positions in the input that never receive gradient (they are never the argmax). Average pool always updates every position, but its gradient signal is diluted by the window size.

---
## Bonus — End-to-end MNIST CNN

The math we just covered all comes together when training a real CNN. Below we build a small CNN, train it on MNIST for a few epochs, and inspect the learned filters.

This section is **optional** — if you skip the cells below, the rest of the notebook is self-contained.


In [ ]:
# Load MNIST
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# transforms.ToTensor() already normalizes pixel values to [0, 1].
transform = transforms.ToTensor()
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

print(f"Training:  {len(train_ds):>6,} samples")
print(f"Test:      {len(test_ds):>6,} samples")
print(f"One image: {train_ds[0][0].shape}  (channels, height, width)")


In [ ]:
# Small CNN: two conv blocks (Conv -> BN -> ReLU -> MaxPool) + classifier head.
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Block 1:  1x28x28 -> 16x14x14
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)
        # Block 2: 16x14x14 -> 32x7x7
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        # Classifier: 32*7*7 -> 64 -> 10
        self.fc1     = nn.Linear(32 * 7 * 7, 64)
        self.fc2     = nn.Linear(64, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.bn1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.bn2(self.conv2(x))), 2)
        x = x.flatten(1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

cnn = SmallCNN()
print(cnn)
print(f"\\nTotal parameters: {count_params(cnn):,}")


In [ ]:
# Train for 3 epochs (enough to hit ~99% accuracy on MNIST)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = cnn.to(device)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f"Training on {device}")
print("-" * 60)
n_epochs = 3
for epoch in range(1, n_epochs + 1):
    # Train
    cnn.train()
    train_loss = 0.0
    for X_, y_ in train_loader:
        X_, y_ = X_.to(device), y_.to(device)
        optimizer.zero_grad()
        loss = criterion(cnn(X_), y_)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Evaluate
    cnn.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_, y_ in test_loader:
            X_, y_ = X_.to(device), y_.to(device)
            preds = cnn(X_).argmax(dim=1)
            correct += (preds == y_).sum().item()
            total   += y_.size(0)

    print(f"Epoch {epoch}/{n_epochs}: "
          f"train loss {train_loss / len(train_loader):.4f}  "
          f"test accuracy {100 * correct / total:.2f}%")


In [ ]:
# Show 8 sample predictions
cnn.eval()
X_sample, y_sample = next(iter(test_loader))
X_sample, y_sample = X_sample[:8], y_sample[:8]
with torch.no_grad():
    preds = cnn(X_sample.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_sample[i].squeeze(), cmap="gray")
    correct = preds[i].item() == y_sample[i].item()
    ax.set_title(f"true: {y_sample[i].item()}, pred: {preds[i].item()}",
                 color="black" if correct else "red")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the learned first-layer filters (16 filters, each 3x3, 1 channel)
filters = cnn.conv1.weight.data.cpu().numpy()  # (16, 1, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(filters[i, 0], cmap="RdBu_r")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Learned 3x3 filters in conv1 (red = positive weight, blue = negative)")
plt.tight_layout()
plt.show()

print("Some filters look like edge detectors --- exactly the patterns")
print("we hand-designed in Problem 4. The network rediscovered them from data.")


---
## Done

You have now seen, in code, every concept covered by the homework PDF:

- Convolution from first principles, verified against PyTorch
- The output-size formula across many cases
- Hand-designed and learned edge detectors
- Padding modes
- Receptive field calculation through real architectures
- Pooling (max and average), both forward and backward
- LeNet-5 reconstructed exactly
- Why $3 \times 3$ filters dominate
- Backprop through conv and pool layers, manually and with autograd
- A real CNN trained on MNIST that converges to ~99% accuracy in three epochs

Cross-check the numerical answers in this notebook against the PDF's solution boxes — they should match exactly.
